# Bibliotekos

In [9]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

# Duomenų paruošimas

In [5]:
file_path = os.path.join("..", "1_laboratorinis", "loan_data.csv")
data = pd.read_csv(file_path)

cols_to_factor = ['credit.policy', 'not.fully.paid', 'purpose']
for col in cols_to_factor:
    data[col] = data[col].astype('category')

data['annual.inc'] = np.exp(data['log.annual.inc'])

print(f"Eilučių skaičius: {len(data)}")
print(f"Stulpelių skaičius: {len(data.columns)}")

Eilučių skaičius: 9578
Stulpelių skaičius: 15


In [51]:
sampled_data = data.groupby(['purpose', 'credit.policy'], observed=False).sample(
    frac=0.6, 
    random_state=6202
).reset_index(drop=True)

print(f"\nAtrinktų duomenų dydis: {len(sampled_data)}")


Atrinktų duomenų dydis: 5746


# Normavimas

In [52]:
def min_max_normalization(x):
    return (x - x.min()) / (x.max() - x.min())

def denormalize(df_norm, orig_min, orig_max):
    return df_norm * (orig_max - orig_min) + orig_min

numeric_cols = sampled_data.select_dtypes(include=[np.number])

orig_min = numeric_cols.min()
orig_max = numeric_cols.max()

min_max_data = numeric_cols.apply(min_max_normalization)

min_max_data  = min_max_data.drop(['int.rate', 'installment', 'log.annual.inc'], axis=1)

categorical_cols = sampled_data.select_dtypes(exclude=[np.number])
full_min_max_data = pd.concat([min_max_data, categorical_cols], axis=1)

min_max_data.head()

,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,annual.inc
0,0.088433,0.404762,0.252568,0.001987,0.901408,0.000000,0.0,0.0,0.015852
1,0.034633,0.285714,0.199361,0.002988,0.413146,0.129032,0.0,0.0,0.050594
2,0.504371,0.214286,0.000000,0.000000,0.448826,0.032258,0.0,0.0,0.001032
3,0.336247,0.333333,0.218260,0.000000,0.203756,0.129032,0.0,0.0,0.087396
4,0.187962,0.547619,0.408971,0.003766,0.646948,0.193548,0.0,0.0,0.050684


In [59]:
debt_data = full_min_max_data[full_min_max_data['purpose'] == 'debt_consolidation'].reset_index()
credit_data = full_min_max_data[full_min_max_data['purpose'] == 'credit_card'].reset_index()

In [60]:
X_train_debt, X_test_debt, y_train_debt, y_test_debt = train_test_split(debt_data.iloc[:, :-3], debt_data['credit.policy'], test_size=0.2, random_state=123)

X_train_credit, X_test_credit, y_train_credit, y_test_credit = train_test_split(credit_data.iloc[:, :-3], credit_data['credit.policy'], test_size=0.2, random_state=123)

# Algoritmai su numatytais parametrais

## SVM